# Model A v2 — Burnout Classification (No Leakage)

**Approach**: Classification instead of regression — predict burnout **category** directly.

**Key design decisions:**
1. **3 classes**: Healthy (score<4), Mildly Burnout (4-7), Burnout (>=7)
2. **10 features** — NO leakage columns
3. **Class imbalance** handled: SMOTE + class_weight='balanced'
4. **Primary metric**: Macro F1 (not accuracy — dummy gets 88.6%)
5. **Optuna** tuning optimizes macro F1
6. **Stacking** ensemble with diverse classifiers
7. **predict_proba** for app gauge pseudo-score

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("All imports OK.")

All imports OK.


## 1. Load & Clean Data

In [2]:
df = pd.read_csv("../../Datasets/academic_stress_level.csv")
print(f"Raw dataset: {len(df):,} rows")

# 10 base features — NO leakage columns
feature_cols = [
    "study_hours_per_day", "sleep_hours", "exam_pressure", "stress_level",
    "financial_stress", "social_support", "anxiety_score", "depression_score",
    "family_expectation", "physical_activity",
]

# Bin burnout_score into 3 classes
CLASS_NAMES = {0: "Healthy", 1: "Mildly Burnout", 2: "Burnout"}

def bin_burnout(score):
    if score < 4:   return 0
    elif score < 7: return 1
    else:           return 2

df["burnout_class"] = df["burnout_score"].apply(bin_burnout)

df_model = df[feature_cols + ["burnout_class"]].dropna()
print(f"After feature selection: {df_model.shape}")
print(f"\nClass distribution:")
for cls, name in CLASS_NAMES.items():
    cnt = (df_model["burnout_class"] == cls).sum()
    print(f"  {cls} ({name:15s}): {cnt:>10,} ({cnt/len(df_model)*100:5.1f}%)")

Raw dataset: 1,000,000 rows
After feature selection: (1000000, 11)

Class distribution:
  0 (Healthy        ):    886,497 ( 88.6%)
  1 (Mildly Burnout ):    109,563 ( 11.0%)
  2 (Burnout        ):      3,940 (  0.4%)


In [3]:
# IQR cleaning on FEATURES ONLY — preserve all class samples
print(f"Before cleaning: {len(df_model):,} rows")

for col in feature_cols:
    q1 = df_model[col].quantile(0.25)
    q3 = df_model[col].quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        continue
    mask = (df_model[col] >= q1 - 1.5 * iqr) & (df_model[col] <= q3 + 1.5 * iqr)
    df_model = df_model[mask]

print(f"After IQR on features only: {len(df_model):,} rows")
print(f"\nClass distribution after cleaning:")
for cls, name in CLASS_NAMES.items():
    cnt = (df_model["burnout_class"] == cls).sum()
    print(f"  {cls} ({name:15s}): {cnt:>10,} ({cnt/len(df_model)*100:5.1f}%)")

Before cleaning: 1,000,000 rows
After IQR on features only: 981,357 rows

Class distribution after cleaning:
  0 (Healthy        ):    877,184 ( 89.4%)
  1 (Mildly Burnout ):    102,024 ( 10.4%)
  2 (Burnout        ):      2,149 (  0.2%)


## 2. Feature Engineering

In [4]:
def engineer_features_a(df):
    """Feature engineering for Model A. Applied to both training and app inference."""
    df = df.copy()
    # Interaction terms
    df["stress_x_anxiety"]     = df["stress_level"] * df["anxiety_score"]
    df["stress_x_depression"]  = df["stress_level"] * df["depression_score"]
    df["stress_x_exam"]        = df["stress_level"] * df["exam_pressure"]
    df["anxiety_x_depression"] = df["anxiety_score"] * df["depression_score"]
    # Ratio features
    df["study_sleep_ratio"]    = df["study_hours_per_day"] / (df["sleep_hours"] + 1)
    df["pressure_support_gap"] = ((df["exam_pressure"] + df["financial_stress"] + df["family_expectation"]) / 3) - df["social_support"]
    # Squared terms
    df["stress_level_sq"]      = df["stress_level"] ** 2
    df["anxiety_score_sq"]     = df["anxiety_score"] ** 2
    # Threshold flags
    df["sleep_deprived"]       = (df["sleep_hours"] < 5).astype(int)
    df["high_stress"]          = (df["stress_level"] > 7).astype(int)
    return df

df_model = engineer_features_a(df_model)
all_features = [c for c in df_model.columns if c != "burnout_class"]
print(f"Total features after engineering: {len(all_features)}")
print(f"Features: {all_features}")

Total features after engineering: 20
Features: ['study_hours_per_day', 'sleep_hours', 'exam_pressure', 'stress_level', 'financial_stress', 'social_support', 'anxiety_score', 'depression_score', 'family_expectation', 'physical_activity', 'stress_x_anxiety', 'stress_x_depression', 'stress_x_exam', 'anxiety_x_depression', 'study_sleep_ratio', 'pressure_support_gap', 'stress_level_sq', 'anxiety_score_sq', 'sleep_deprived', 'high_stress']


## 3. Train/Test Split + SMOTE

In [5]:
X = df_model[all_features]
y = df_model["burnout_class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"\nTrain class distribution:")
for cls, name in CLASS_NAMES.items():
    cnt = (y_train == cls).sum()
    print(f"  {cls} ({name:15s}): {cnt:>10,} ({cnt/len(y_train)*100:5.1f}%)")

# Apply SMOTE to training data only
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"\nAfter SMOTE:")
for cls, name in CLASS_NAMES.items():
    cnt = (y_train_sm == cls).sum()
    print(f"  {cls} ({name:15s}): {cnt:>10,} ({cnt/len(y_train_sm)*100:5.1f}%)")

Train: 785,085 | Test: 196,272

Train class distribution:
  0 (Healthy        ):    701,747 ( 89.4%)
  1 (Mildly Burnout ):     81,619 ( 10.4%)
  2 (Burnout        ):      1,719 (  0.2%)

After SMOTE:
  0 (Healthy        ):    701,747 ( 33.3%)
  1 (Mildly Burnout ):    701,747 ( 33.3%)
  2 (Burnout        ):    701,747 ( 33.3%)


## 4. Baseline Comparison (Macro F1)

Primary metric: **Macro F1** — treats all 3 classes equally regardless of size.
A dummy classifier that always predicts "Healthy" gets macro F1 ~0.31.

In [6]:
# Subsample SMOTE data — reused for BOTH baseline CV AND Optuna tuning
# Without this, cross_val_score on 2.1M rows with n_jobs=-1 causes OOM
TUNE_SIZE = 200_000
if len(X_train_sm) > TUNE_SIZE:
    idx = np.random.RandomState(42).choice(len(X_train_sm), TUNE_SIZE, replace=False)
    X_tune = X_train_sm.iloc[idx].reset_index(drop=True)
    y_tune = y_train_sm.iloc[idx].reset_index(drop=True)
    print(f"Subsampled to {TUNE_SIZE:,} rows for CV (from {len(X_train_sm):,} SMOTE rows)")
else:
    X_tune, y_tune = X_train_sm, y_train_sm
    print(f"Using full {len(X_tune):,} SMOTE rows for CV")

print(f"\nClass distribution in subsample:")
for cls, name in CLASS_NAMES.items():
    cnt = (y_tune == cls).sum()
    print(f"  {cls} ({name:15s}): {cnt:>8,} ({cnt/len(y_tune)*100:5.1f}%)")

Subsampled to 200,000 rows for CV (from 2,105,241 SMOTE rows)

Class distribution in subsample:
  0 (Healthy        ):   66,830 ( 33.4%)
  1 (Mildly Burnout ):   66,456 ( 33.2%)
  2 (Burnout        ):   66,714 ( 33.4%)


In [7]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baselines = {
    "LightGBM":    lgb.LGBMClassifier(n_estimators=800, class_weight="balanced", random_state=42, verbosity=-1),
    "XGBoost":     xgb.XGBClassifier(n_estimators=800, random_state=42, verbosity=0),
    "CatBoost":    CatBoostClassifier(iterations=800, auto_class_weights="Balanced", random_seed=42, verbose=0, allow_writing_files=False),
    "RandomForest":RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42, n_jobs=-1),
}

print("5-Fold Stratified CV Baseline (200K subsample, macro F1):")
print("=" * 60)
baseline_scores = {}
for name, model in baselines.items():
    scores = cross_val_score(model, X_tune, y_tune, cv=skf, scoring="f1_macro", n_jobs=-1)
    baseline_scores[name] = scores.mean()
    print(f"  {name:15s} | Macro F1 = {scores.mean():.4f} +/- {scores.std():.4f}")

print(f"\n>>> Dummy (always Healthy): Macro F1 ~ 0.31")
print(f">>> Best baseline:          Macro F1 = {max(baseline_scores.values()):.4f} ({max(baseline_scores, key=baseline_scores.get)})")

5-Fold Stratified CV Baseline (200K subsample, macro F1):
  LightGBM        | Macro F1 = 0.8998 +/- 0.0018
  XGBoost         | Macro F1 = 0.9079 +/- 0.0021
  CatBoost        | Macro F1 = 0.8767 +/- 0.0029
  RandomForest    | Macro F1 = 0.9129 +/- 0.0020

>>> Dummy (always Healthy): Macro F1 ~ 0.31
>>> Best baseline:          Macro F1 = 0.9129 (RandomForest)


## 5. Optuna Hyperparameter Tuning (Optimizing Macro F1)

Tuning on 200K-row SMOTE-balanced subsample with StratifiedKFold.

In [8]:
# X_tune, y_tune already defined above (TUNE_SIZE = 200,000) — shared with baseline CV
print(f"Tuning on {len(X_tune):,} rows (shared 200K subsample from Section 4)")
skf_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ─── LightGBM ───
def objective_lgbm(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 500, 3000),
        "max_depth":         trial.suggest_int("max_depth", 4, 12),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 31, 255),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "class_weight": "balanced", "verbosity": -1, "random_state": 42,
    }
    model = lgb.LGBMClassifier(**params)
    scores = cross_val_score(model, X_tune, y_tune, cv=skf_tune, scoring="f1_macro", n_jobs=-1)
    return scores.mean()

print("Tuning LightGBM (150 trials, macro F1)...")
study_lgbm = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, n_trials=150, show_progress_bar=True)
print(f"\nBest LightGBM Macro F1: {study_lgbm.best_value:.4f}")
print(f"Best params: {study_lgbm.best_params}")

Tuning on 200,000 rows (shared 200K subsample from Section 4)
Tuning LightGBM (150 trials, macro F1)...


Best trial: 0. Best value: 0.913578:   3%|▎         | 5/150 [10:11<4:55:20, 122.21s/it]


[W 2026-05-13 10:30:02,660] Trial 5 failed with parameters: {'n_estimators': 2156, 'max_depth': 6, 'learning_rate': 0.023746198818402044, 'num_leaves': 154, 'min_child_samples': 22, 'subsample': 0.9878338511058234, 'colsample_bytree': 0.8875664116805573, 'reg_alpha': 5.727904470799623, 'reg_lambda': 3.7958531426706403} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "e:\BINUS\Semester 4\Machine Learning\AOL\Academic-Shield-AI-Based-Burnout-Meter-and-Student-Performance-Predictor\venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\User\AppData\Local\Temp\ipykernel_18988\4294934651.py", line 20, in objective_lgbm
    scores = cross_val_score(model, X_tune, y_tune, cv=skf_tune, scoring="f1_macro", n_jobs=-1)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\BINUS\Semester 4\Machin

KeyboardInterrupt: 

In [ ]:
# ─── XGBoost ───
def objective_xgb(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 500, 3000),
        "max_depth":         trial.suggest_int("max_depth", 4, 12),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "min_child_weight":  trial.suggest_int("min_child_weight", 1, 50),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "verbosity": 0, "random_state": 42,
    }
    model = xgb.XGBClassifier(**params)
    scores = cross_val_score(model, X_tune, y_tune, cv=skf_tune, scoring="f1_macro", n_jobs=-1)
    return scores.mean()

print("Tuning XGBoost (150 trials, macro F1)...")
study_xgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=150, show_progress_bar=True)
print(f"\nBest XGBoost Macro F1: {study_xgb.best_value:.4f}")
print(f"Best params: {study_xgb.best_params}")

In [ ]:
# ─── CatBoost ───
def objective_cat(trial):
    params = {
        "iterations":        trial.suggest_int("iterations", 500, 3000),
        "depth":             trial.suggest_int("depth", 4, 10),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "l2_leaf_reg":       trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "auto_class_weights": "Balanced",
        "random_seed": 42, "verbose": 0, "allow_writing_files": False,
    }
    model = CatBoostClassifier(**params)
    scores = cross_val_score(model, X_tune, y_tune, cv=skf_tune, scoring="f1_macro", n_jobs=-1)
    return scores.mean()

print("Tuning CatBoost (150 trials, macro F1)...")
study_cat = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_cat.optimize(objective_cat, n_trials=150, show_progress_bar=True)
print(f"\nBest CatBoost Macro F1: {study_cat.best_value:.4f}")
print(f"Best params: {study_cat.best_params}")

## 6. Retrain Tuned Classifiers on SMOTE Training Data & Evaluate on Original Test Set

Train each classifier with Optuna-tuned parameters on the SMOTE-balanced training data, then evaluate on the **original** (imbalanced) test set — this reflects real-world performance.

In [ ]:
# Build best classifiers with Optuna params, train on SMOTE data, evaluate on original test
best_lgbm = lgb.LGBMClassifier(**study_lgbm.best_params, class_weight="balanced", random_state=42, verbosity=-1)
best_xgb  = xgb.XGBClassifier(**study_xgb.best_params, random_state=42, verbosity=0)

cat_p = study_cat.best_params.copy()
cat_p["random_seed"] = 42
cat_p["verbose"] = 0
cat_p["allow_writing_files"] = False
cat_p["auto_class_weights"] = "Balanced"
best_cat = CatBoostClassifier(**cat_p)

print("Training on SMOTE data, evaluating on ORIGINAL test set:")
print("=" * 70)
individual_results = {}
for name, model in [("LightGBM", best_lgbm), ("XGBoost", best_xgb), ("CatBoost", best_cat)]:
    model.fit(X_train_sm, y_train_sm)
    pred = model.predict(X_test)
    acc  = accuracy_score(y_test, pred)
    f1_m = f1_score(y_test, pred, average="macro")
    individual_results[name] = {"model": model, "Accuracy": acc, "Macro_F1": f1_m, "preds": pred}
    print(f"\n  {name}")
    print(f"  Accuracy = {acc:.4f} | Macro F1 = {f1_m:.4f}")
    print(classification_report(y_test, pred, target_names=list(CLASS_NAMES.values()), digits=4))

## 7. Stacking Classifier Ensemble

Base learners: LightGBM + XGBoost + CatBoost (all tuned).  
Meta learner: LogisticRegression(class_weight='balanced') — learns optimal class combination.

In [ ]:
# Fresh instances for stacking (cannot reuse already-fitted models)
stack_lgbm = lgb.LGBMClassifier(**study_lgbm.best_params, class_weight="balanced", random_state=42, verbosity=-1)
stack_xgb  = xgb.XGBClassifier(**study_xgb.best_params, random_state=42, verbosity=0)

cat_p2 = study_cat.best_params.copy()
cat_p2["random_seed"] = 42
cat_p2["verbose"] = 0
cat_p2["allow_writing_files"] = False
cat_p2["auto_class_weights"] = "Balanced"
stack_cat = CatBoostClassifier(**cat_p2)

stacking = StackingClassifier(
    estimators=[
        ("lgbm", stack_lgbm),
        ("xgb",  stack_xgb),
        ("cat",  stack_cat),
    ],
    final_estimator=LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    stack_method="predict_proba",
    n_jobs=-1,
)

print("Fitting stacking classifier on SMOTE training data...")
stacking.fit(X_train_sm, y_train_sm)
pred_stack = stacking.predict(X_test)

acc_s  = accuracy_score(y_test, pred_stack)
f1_s   = f1_score(y_test, pred_stack, average="macro")
print(f"\n  Stacking Classifier")
print(f"  Accuracy = {acc_s:.4f} | Macro F1 = {f1_s:.4f}")
print(classification_report(y_test, pred_stack, target_names=list(CLASS_NAMES.values()), digits=4))

## 8. Final Comparison & Select Best Model

Primary metric: **Macro F1** (not accuracy — dummy majority-class baseline gets ~88.6% accuracy but only ~0.31 macro F1).

In [ ]:
# Compare all models
all_results = {}
for k, v in individual_results.items():
    all_results[k] = {"Accuracy": v["Accuracy"], "Macro_F1": v["Macro_F1"]}
all_results["Stacking"] = {"Accuracy": acc_s, "Macro_F1": f1_s}

results_df = pd.DataFrame(all_results).T.sort_values("Macro_F1", ascending=False)

print("=" * 65)
print("     FINAL MODEL COMPARISON — Classification (Test Set)")
print("=" * 65)
print(results_df.to_string())
print(f"\n  Dummy (always Healthy): Accuracy ~88.6%, Macro F1 ~0.31")
print("=" * 65)

best_name = results_df.index[0]
print(f"\n>>> BEST MODEL: {best_name}")
print(f"    Accuracy = {results_df.loc[best_name, 'Accuracy']:.4f}")
print(f"    Macro F1 = {results_df.loc[best_name, 'Macro_F1']:.4f}")

# Confusion matrix for best model
if best_name == "Stacking":
    best_preds = pred_stack
else:
    best_preds = individual_results[best_name]["preds"]

fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, best_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(CLASS_NAMES.values()))
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title(f"Confusion Matrix — {best_name} (Best Model)")
plt.tight_layout()
plt.show()

In [ ]:
# 5-Fold Stratified CV on best model using FULL dataset
if best_name == "Stacking":
    cv_lgbm = lgb.LGBMClassifier(**study_lgbm.best_params, class_weight="balanced", random_state=42, verbosity=-1)
    cv_xgb  = xgb.XGBClassifier(**study_xgb.best_params, random_state=42, verbosity=0)
    cat_cv_p = study_cat.best_params.copy()
    cat_cv_p["random_seed"] = 42; cat_cv_p["verbose"] = 0; cat_cv_p["allow_writing_files"] = False
    cat_cv_p["auto_class_weights"] = "Balanced"
    cv_cat = CatBoostClassifier(**cat_cv_p)
    best_final = StackingClassifier(
        estimators=[("lgbm", cv_lgbm), ("xgb", cv_xgb), ("cat", cv_cat)],
        final_estimator=LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        stack_method="predict_proba", n_jobs=-1,
    )
else:
    best_final = individual_results[best_name]["model"]

# NOTE: CV on original (non-SMOTE) data to see real-world generalization
skf_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Running 5-Fold Stratified CV on FULL dataset for {best_name}...")
cv_acc = cross_val_score(best_final, X, y, cv=skf_final, scoring="accuracy", n_jobs=-1)
cv_f1  = cross_val_score(best_final, X, y, cv=skf_final, scoring="f1_macro", n_jobs=-1)

print(f"\n5-Fold Stratified CV of {best_name}:")
print(f"  CV Accuracy : {cv_acc.mean():.4f} +/- {cv_acc.std():.4f}")
print(f"  CV Macro F1 : {cv_f1.mean():.4f} +/- {cv_f1.std():.4f}")
print(f"\n  Dummy baseline: Accuracy ~0.886, Macro F1 ~0.31")

## 9. Export Model, Pipeline & Class Mapping

Saves:
1. **modelA.pkl** — best classifier (supports `predict` + `predict_proba`)
2. **pipeline_a.pkl** — feature engineering function
3. **burnout_class_mapping.pkl** — {0: "Healthy", 1: "Mildly Burnout", 2: "Burnout"}

In [ ]:
import os
os.makedirs("../models", exist_ok=True)

# Pick the best model
if best_name == "Stacking":
    final_model = stacking
else:
    final_model = individual_results[best_name]["model"]

# Save classifier model
joblib.dump(final_model, "../models/modelA.pkl")
print(f"Saved {best_name} classifier to ../models/modelA.pkl")

# Save feature engineering function as pipeline
joblib.dump(engineer_features_a, "../models/pipeline_a.pkl")
print(f"Saved engineer_features_a to ../models/pipeline_a.pkl")

# Save class mapping
joblib.dump(CLASS_NAMES, "../models/burnout_class_mapping.pkl")
print(f"Saved class mapping to ../models/burnout_class_mapping.pkl")
print(f"  {CLASS_NAMES}")

In [ ]:
# Verification — simulate app input (NO leakage columns, only 10 base features)
loaded_model   = joblib.load("../models/modelA.pkl")
loaded_pipe    = joblib.load("../models/pipeline_a.pkl")
loaded_classes = joblib.load("../models/burnout_class_mapping.pkl")

print(f"Class mapping: {loaded_classes}")
print(f"Model type: {type(loaded_model).__name__}")
print(f"Has predict_proba: {hasattr(loaded_model, 'predict_proba')}")

test_input = pd.DataFrame([{
    "study_hours_per_day": 5.0, "sleep_hours": 7.0, "exam_pressure": 6.0,
    "stress_level": 4.5, "financial_stress": 5.0, "social_support": 5.0,
    "anxiety_score": 3.0, "depression_score": 2.0,
    "family_expectation": 6.0, "physical_activity": 2.0,
}])

test_engineered = loaded_pipe(test_input)
pred_class = loaded_model.predict(test_engineered)[0]
pred_proba = loaded_model.predict_proba(test_engineered)[0]

# Pseudo-score for app gauge (0-100)
pseudo_score = pred_proba[0] * 20 + pred_proba[1] * 55 + pred_proba[2] * 85

print(f"\nTest prediction:")
print(f"  Class:        {pred_class} ({loaded_classes[pred_class]})")
print(f"  Probabilities: Healthy={pred_proba[0]:.3f}, Mild={pred_proba[1]:.3f}, Burnout={pred_proba[2]:.3f}")
print(f"  Pseudo-score:  {pseudo_score:.1f}/100 (for gauge)")

# Edge cases
print("\n--- Edge Cases ---")
edge_cases = [
    {"label": "Low burnout",  "stress_level": 1.0, "anxiety_score": 1.0, "depression_score": 1.0,
     "exam_pressure": 2.0, "financial_stress": 1.0, "family_expectation": 2.0,
     "social_support": 9.0, "sleep_hours": 9.0, "study_hours_per_day": 3.0, "physical_activity": 5.0},
    {"label": "High burnout", "stress_level": 9.5, "anxiety_score": 9.0, "depression_score": 9.0,
     "exam_pressure": 9.0, "financial_stress": 9.0, "family_expectation": 9.0,
     "social_support": 1.0, "sleep_hours": 3.0, "study_hours_per_day": 12.0, "physical_activity": 0.5},
    {"label": "Mid range",    "stress_level": 5.0, "anxiety_score": 5.0, "depression_score": 5.0,
     "exam_pressure": 5.0, "financial_stress": 5.0, "family_expectation": 5.0,
     "social_support": 5.0, "sleep_hours": 6.0, "study_hours_per_day": 5.0, "physical_activity": 2.0},
]

for case in edge_cases:
    label = case.pop("label")
    row = pd.DataFrame([case])
    row_eng = loaded_pipe(row)
    cls = loaded_model.predict(row_eng)[0]
    prb = loaded_model.predict_proba(row_eng)[0]
    ps  = prb[0] * 20 + prb[1] * 55 + prb[2] * 85
    print(f"  {label:15s} -> {loaded_classes[cls]:15s} | proba=[{prb[0]:.3f}, {prb[1]:.3f}, {prb[2]:.3f}] | gauge={ps:.1f}/100")

print("\nVerification PASSED — Model A is now a classifier with predict_proba.")